In [6]:
# ==================== Final re-runs from a saved study ====================

def rerun_from_saved_study(
   
    objective_fn,
    epochs_list=(50, 75, 100),
    batch_list=(16, 32),
    seed: int = 42,
    tag_prefix: str = "FINAL",
):
    """
    Reload an existing Optuna study (sqlite under `scenario_dir/optuna_study.db`),
    take its best params, and run a grid of final trainings with custom epochs/batch sizes.

    Each run is executed by calling:
        objective_fn(FixedTrial(best_params),
                     force_epochs=<E>,
                     force_batch=<B>,
                     save_tag=f"{tag_prefix}_e{E}_bs{B}")

    Artifacts are stored under the same scenario directory your objective uses.
    """
    
    db_path = "optuna_study.db"
    if not db_path.exists():
        raise FileNotFoundError(f"No study DB found at: {db_path}")

    # Re-load the study
    study = optuna.load_study(
        study_name=None,  # default study in that DB
        storage=f"sqlite:///{db_path}"
    )

    if study.best_trial is None or study.best_params is None:
        raise RuntimeError("The loaded study has no best trial/params.")

    best_params = study.best_params.copy()
    print("══════════ Loaded best params from saved study ══════════")
    print(best_params)

    # Seed reproducibility
    tf.random.set_seed(seed)
    set_seed(seed)

    # Run grid of (epochs x batch_size)
    results = []
    for E in epochs_list:
        for B in batch_list:
            tag = f"{tag_prefix}_e{E}_bs{B}"
            print(f"\n▶︎ Re-training with best params | epochs={E}, batch={B} | tag={tag}")

            try:
                fixed = optuna.trial.FixedTrial(best_params)
                score = objective_fn(
                    fixed,
                    force_epochs=int(E),
                    force_batch=int(B),
                    save_tag=tag
                )
                print(f"✅  Done: {tag}  | optuna_score={score:.6f}")
                results.append({"epochs": E, "batch": B, "score": float(score), "tag": tag})
            except Exception as e:
                print(f"⚠️  Failed run for E={E}, B={B}: {e}")
            finally:
                # keep runs isolated
                tf.keras.backend.clear_session()
                gc.collect()

    # Save a small summary CSV next to the study DB
    if results:
        out_df = pd.DataFrame(results).sort_values("score", ascending=(study.direction.name=="MINIMIZE"))
        out_csv = scenario_dir / "final_grid_results.csv"
        out_df.to_csv(out_csv, index=False)
        print(f"\n📄 Saved final grid summary → {out_csv}")
        print(out_df)
    else:
        print("\n⚠️  No successful final runs recorded.")


# ---------------- Example: call this after your `run_optuna(...)` or as a separate step ----------------
if __name__ == "__main__":
    # ... your existing argparse above ...

    # After you run scenarios (or when you want to repeat),
    # call the helper like this (adjust the path to your scenario/tag directory):
    #
       rerun_from_saved_study(
         
           objective_fn=objective_fn,            # the same partial(...) you already build in run_scenario
           epochs_list=[50, 80, 100, 150],
           batch_list=[8, 16, 32],
           seed=42,
           tag_prefix="FINAL_RERUN"
       )
    



NameError: name 'objective_fn' is not defined